# Spectral Refinement via Peak-Weighted RMSE + REINFORCE F1

Post-hoc spectral refinement network trained on SpectraLoRA's predicted spectra.

**Architecture**: 1D U-Net with FiLM conditioning (Morgan FP)

**Training (3 phases)**:
1. Peak-weighted RMSE warm-up (Mol2Raman loss)
2. Sinkhorn soft F1 (differentiable)
3. REINFORCE hard F1 (non-differentiable, policy gradient)

In [1]:
from __future__ import annotations
import sys, json, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import find_peaks
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings('ignore')

REPO_ROOT = Path('/Users/rahul/Desktop/hp-proteins-ml')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'capsule-3259363' / 'code'))

DEVICE = 'cpu'
print('OK')

OK


In [2]:
# === Load cached data ===
CACHE_DIR = REPO_ROOT / 'ramanchembl_pipeline' / 'artifacts' / 'alignment' / 'cache'

data = np.load(CACHE_DIR / 'dft_point_v1_1000.npz', allow_pickle=True)
meta = pd.read_csv(CACHE_DIR / 'dft_point_v1_1000.csv')

y_pred = data['y_pred_spec'].astype(np.float32)      # (N, 3501) predicted broadened spectra
y_target = data['y_target_spec'].astype(np.float32)   # (N, 3501) DFT reference broadened spectra
x_grid = data['x_grid'].astype(np.float64)            # (3501,)
target_freq = data['target_freq']                      # (N, max_modes)
target_mask = data['target_mask']                      # (N, max_modes) bool

# Normalize spectra to [0, 1] per molecule
def normalize_spectra(specs):
    mx = specs.max(axis=1, keepdims=True)
    mx = np.where(mx > 0, mx, 1.0)
    return specs / mx

y_pred_norm = normalize_spectra(y_pred)
y_target_norm = normalize_spectra(y_target)

print(f'Loaded {len(y_pred)} molecules, grid={len(x_grid)} points [{x_grid[0]:.0f}, {x_grid[-1]:.0f}] cm⁻¹')
print(f'Pred spec range: [{y_pred_norm.min():.3f}, {y_pred_norm.max():.3f}]')
print(f'Target spec range: [{y_target_norm.min():.3f}, {y_target_norm.max():.3f}]')

Loaded 1000 molecules, grid=3501 points [500, 4000] cm⁻¹
Pred spec range: [0.000, 1.000]
Target spec range: [0.000, 1.000]


In [3]:
# === Compute Morgan fingerprints ===
from rdkit import Chem
from rdkit.Chem import AllChem

def compute_morgan_fps(smiles_list, n_bits=2048, radius=2):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            fps.append(np.zeros(n_bits, dtype=np.float32))
        else:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(np.array(fp, dtype=np.float32))
    return np.stack(fps)

smiles_col = 'smiles' if 'smiles' in meta.columns else 'SMILES'
morgan_fps = compute_morgan_fps(meta[smiles_col].tolist())
print(f'Morgan FPs: {morgan_fps.shape}')

[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerator
[16:10:32] DEPRECATION WARNING: please use MorganGenerat

Morgan FPs: (1000, 2048)


In [4]:
# === Train/val/test split ===
N = len(y_pred_norm)
rng = np.random.default_rng(42)
indices = rng.permutation(N)
n_train = int(0.7 * N)
n_val = int(0.15 * N)
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]
print(f'Split: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')

class SpectrumDataset(Dataset):
    def __init__(self, pred, target, morgan, target_freq, target_mask, indices):
        self.pred = torch.from_numpy(pred[indices])
        self.target = torch.from_numpy(target[indices])
        self.morgan = torch.from_numpy(morgan[indices])
        self.target_freq = target_freq[indices]
        self.target_mask = target_mask[indices]
    def __len__(self): return len(self.pred)
    def __getitem__(self, i):
        return self.pred[i], self.target[i], self.morgan[i], i

train_ds = SpectrumDataset(y_pred_norm, y_target_norm, morgan_fps, target_freq, target_mask, train_idx)
val_ds = SpectrumDataset(y_pred_norm, y_target_norm, morgan_fps, target_freq, target_mask, val_idx)
test_ds = SpectrumDataset(y_pred_norm, y_target_norm, morgan_fps, target_freq, target_mask, test_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
print('Dataloaders OK')

Split: train=700, val=150, test=150
Dataloaders OK


In [5]:
# === Peak-Weighted RMSE Loss (Mol2Raman paper, eq. 1) ===

def peak_weighted_rmse(pred, target, threshold=0.5, temperature=0.05,
                       w_tp=8.0, w_fp=6.0, w_fn=5.0, w_tn=1.0):
    """Differentiable peak-weighted RMSE from Mol2Raman.
    Uses soft sigmoid thresholding for gradient flow."""
    # Soft peak masks (differentiable)
    target_peak = torch.sigmoid((target - threshold) / temperature)
    pred_peak = torch.sigmoid((pred - threshold) / temperature)
    
    tp = target_peak * pred_peak
    fp = (1 - target_peak) * pred_peak
    fn = target_peak * (1 - pred_peak)
    tn = (1 - target_peak) * (1 - pred_peak)
    
    weights = w_tp * tp + w_fp * fp + w_fn * fn + w_tn * tn
    weighted_sq = weights * (pred - target) ** 2
    return weighted_sq.mean(dim=-1).sqrt().mean()

# Quick test
a = torch.randn(4, 3501).sigmoid()
b = torch.randn(4, 3501).sigmoid()
print(f'Test loss: {peak_weighted_rmse(a, b):.4f}')

Test loss: 0.6853


In [6]:
# === Refinement U-Net with FiLM conditioning ===

class FiLM(nn.Module):
    """Feature-wise Linear Modulation: condition conv features on molecular identity."""
    def __init__(self, cond_dim, n_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, 128), nn.ReLU(),
            nn.Linear(128, 2 * n_channels)
        )
    def forward(self, x, cond):
        # x: (B, C, L), cond: (B, cond_dim)
        params = self.net(cond)  # (B, 2C)
        gamma, beta = params.chunk(2, dim=-1)  # (B, C) each
        return x * (1 + gamma.unsqueeze(-1)) + beta.unsqueeze(-1)

class ResConvBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv1d(ch, ch, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(ch)
        self.conv2 = nn.Conv1d(ch, ch, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(ch)
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        return F.relu(x + h)

class RefinementNet(nn.Module):
    """1D U-Net that learns a residual delta on the input spectrum.
    FiLM-conditioned on Morgan fingerprint at bottleneck."""
    def __init__(self, in_len=3501, cond_dim=2048, dropout=0.15):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(nn.Conv1d(1, 16, 7, padding=3), nn.BatchNorm1d(16), nn.ReLU(), ResConvBlock(16))
        self.enc2 = nn.Sequential(nn.Conv1d(16, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), ResConvBlock(32))
        self.enc3 = nn.Sequential(nn.Conv1d(32, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), ResConvBlock(64))
        self.pool = nn.MaxPool1d(2)
        
        # Bottleneck + FiLM
        self.bottleneck = ResConvBlock(64)
        self.film = FiLM(cond_dim, 64)
        self.drop = nn.Dropout(dropout)
        
        # Decoder
        self.up3 = nn.ConvTranspose1d(64, 64, 2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv1d(128, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), ResConvBlock(32))
        self.up2 = nn.ConvTranspose1d(32, 32, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv1d(64, 16, 5, padding=2), nn.BatchNorm1d(16), nn.ReLU(), ResConvBlock(16))
        self.up1 = nn.ConvTranspose1d(16, 16, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv1d(32, 16, 5, padding=2), nn.BatchNorm1d(16), nn.ReLU(), ResConvBlock(16))
        
        # Head — zero-initialized for residual learning
        self.head = nn.Conv1d(16, 1, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        
        self.in_len = in_len
    
    def forward(self, spectrum, morgan_fp):
        """spectrum: (B, L), morgan_fp: (B, 2048) -> refined: (B, L)"""
        x = spectrum.unsqueeze(1)  # (B, 1, L)
        
        # Pad to even length for pooling
        L = x.shape[-1]
        pad = (8 - L % 8) % 8
        if pad > 0:
            x = F.pad(x, (0, pad))
        
        # Encode
        e1 = self.enc1(x)           # (B, 16, L)
        e2 = self.enc2(self.pool(e1))  # (B, 32, L/2)
        e3 = self.enc3(self.pool(e2))  # (B, 64, L/4)
        
        # Bottleneck + FiLM + dropout
        b = self.bottleneck(self.pool(e3))  # (B, 64, L/8)
        b = self.film(b, morgan_fp)
        b = self.drop(b)
        
        # Decode with skip connections
        d3 = self.up3(b)[:, :, :e3.shape[-1]]
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)[:, :, :e2.shape[-1]]
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)[:, :, :e1.shape[-1]]
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        delta = self.head(d1).squeeze(1)  # (B, L+pad)
        if pad > 0:
            delta = delta[:, :L]
        
        return (spectrum + delta).clamp(0, 1)

model = RefinementNet(in_len=len(x_grid), cond_dim=2048).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'RefinementNet: {n_params:,} parameters')

# Test forward
with torch.no_grad():
    test_out = model(torch.randn(2, len(x_grid)), torch.randn(2, 2048))
print(f'Test forward: {test_out.shape}')

RefinementNet: 447,905 parameters
Test forward: torch.Size([2, 3501])


In [7]:
# === F1 computation (non-differentiable, for REINFORCE reward) ===

def extract_peaks(spectrum, x_grid, prominence=0.02, height=0.01, distance=5):
    """Extract peak positions from a normalized spectrum."""
    peaks, _ = find_peaks(spectrum, prominence=prominence, height=height, distance=distance)
    return x_grid[peaks]

def f1_at_tolerance(ref_peaks, pred_peaks, tol=15.0):
    """F1 via greedy one-to-one matching (Mol2Raman protocol)."""
    if len(ref_peaks) == 0 or len(pred_peaks) == 0:
        return 0.0
    ref, pred = np.sort(ref_peaks), np.sort(pred_peaks)
    matched = set()
    tp = 0
    for p in pred:
        dists = np.abs(ref - p)
        for idx in np.argsort(dists):
            if dists[idx] > tol: break
            if idx not in matched:
                matched.add(idx); tp += 1; break
    prec = tp / len(pred) if len(pred) else 0
    rec = tp / len(ref) if len(ref) else 0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def batch_f1(pred_specs_np, target_specs_np, x_grid_np, tol=15.0):
    """Compute F1@tol for a batch of spectra."""
    f1s = []
    for i in range(len(pred_specs_np)):
        ref_peaks = extract_peaks(target_specs_np[i], x_grid_np)
        pred_peaks = extract_peaks(pred_specs_np[i], x_grid_np)
        f1s.append(f1_at_tolerance(ref_peaks, pred_peaks, tol))
    return np.array(f1s, dtype=np.float32)

print('F1 functions OK')

F1 functions OK


In [8]:
# === REINFORCE loss ===

def reinforce_f1_loss(model, spectrum, morgan_fp, target_spec, x_grid_np,
                      K=5, baseline=None, sigma=0.02, tol=15.0):
    """
    REINFORCE with F1@tol as reward.
    K candidates via dropout stochasticity.
    Returns: loss (scalar), mean_reward (float), new_baseline (float)
    """
    B = spectrum.shape[0]
    
    # Deterministic forward for baseline (mu)
    model.eval()
    with torch.no_grad():
        mu = model(spectrum, morgan_fp)  # (B, L)
    model.train()
    
    all_rewards = []
    all_log_probs = []
    
    target_np = target_spec.cpu().numpy()
    
    for k in range(K):
        # Stochastic forward (dropout active)
        candidate = model(spectrum, morgan_fp)  # (B, L)
        
        # Log-prob under Gaussian policy centered at mu
        log_prob = -0.5 * ((candidate - mu.detach()) ** 2).sum(dim=-1) / (sigma ** 2)  # (B,)
        all_log_probs.append(log_prob)
        
        # Non-differentiable F1 reward
        with torch.no_grad():
            cand_np = candidate.cpu().numpy()
            rewards = batch_f1(cand_np, target_np, x_grid_np, tol=tol)
            all_rewards.append(torch.tensor(rewards, device=spectrum.device))
    
    rewards = torch.stack(all_rewards, dim=0)    # (K, B)
    log_probs = torch.stack(all_log_probs, dim=0) # (K, B)
    
    mean_reward = rewards.mean().item()
    
    # Advantage with baseline
    if baseline is None:
        baseline = mean_reward
    advantage = rewards - baseline  # (K, B)
    
    # REINFORCE: -E[advantage * log_pi]
    loss = -(advantage.detach() * log_probs).mean()
    
    new_baseline = 0.99 * baseline + 0.01 * mean_reward
    return loss, mean_reward, new_baseline

print('REINFORCE OK')

REINFORCE OK


In [9]:
# === Evaluation ===

@torch.no_grad()
def evaluate(model, loader, x_grid_np, tol=15.0):
    model.eval()
    all_pw_rmse = []
    all_f1 = []
    for pred, target, morgan, _ in loader:
        refined = model(pred, morgan)
        pw = peak_weighted_rmse(refined, target).item()
        all_pw_rmse.append(pw)
        
        f1s = batch_f1(refined.numpy(), target.numpy(), x_grid_np, tol=tol)
        all_f1.extend(f1s.tolist())
    
    # Also compute F1 on the UNREFINED input for comparison
    baseline_f1 = []
    for pred, target, morgan, _ in loader:
        f1s = batch_f1(pred.numpy(), target.numpy(), x_grid_np, tol=tol)
        baseline_f1.extend(f1s.tolist())
    
    return {
        'pw_rmse': np.mean(all_pw_rmse),
        'f1': np.mean(all_f1),
        'f1_baseline': np.mean(baseline_f1),
    }

# Baseline before training
x_grid_np = x_grid
baseline_metrics = evaluate(model, val_loader, x_grid_np)
print(f'Before training: F1@15={baseline_metrics["f1"]:.3f} (baseline input: {baseline_metrics["f1_baseline"]:.3f})')

Before training: F1@15=0.366 (baseline input: 0.366)


In [10]:
# === PHASE 1: Peak-Weighted RMSE warm-up ===

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150, eta_min=1.5e-5)

best_val_loss = float('inf')
patience_counter = 0
history = {'phase': [], 'epoch': [], 'train_loss': [], 'val_pw': [], 'val_f1': [], 'val_f1_base': []}

print('=== PHASE 1: Peak-Weighted RMSE ===')
for epoch in range(150):
    model.train()
    epoch_loss = []
    for pred, target, morgan, _ in train_loader:
        refined = model(pred, morgan)
        loss = peak_weighted_rmse(refined, target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss.append(loss.item())
    scheduler.step()
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        metrics = evaluate(model, val_loader, x_grid_np)
        history['phase'].append(1)
        history['epoch'].append(epoch)
        history['train_loss'].append(np.mean(epoch_loss))
        history['val_pw'].append(metrics['pw_rmse'])
        history['val_f1'].append(metrics['f1'])
        history['val_f1_base'].append(metrics['f1_baseline'])
        print(f'  [{epoch+1:3d}] train_loss={np.mean(epoch_loss):.4f} | '
              f'val_pw={metrics["pw_rmse"]:.4f} | '
              f'val_F1@15={metrics["f1"]:.3f} (base={metrics["f1_baseline"]:.3f}) | '
              f'lr={scheduler.get_last_lr()[0]:.1e}')
        
        if metrics['pw_rmse'] < best_val_loss:
            best_val_loss = metrics['pw_rmse']
            torch.save(model.state_dict(), 'refinement_phase1_best.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 25:
                print(f'  Early stopping at epoch {epoch+1}')
                break

# Reload best
model.load_state_dict(torch.load('refinement_phase1_best.pth', weights_only=True))
metrics = evaluate(model, val_loader, x_grid_np)
print(f'\nPhase 1 best: val_pw={metrics["pw_rmse"]:.4f} | F1@15={metrics["f1"]:.3f} (base={metrics["f1_baseline"]:.3f})')

=== PHASE 1: Peak-Weighted RMSE ===
  [  1] train_loss=0.2012 | val_pw=0.2478 | val_F1@15=0.366 (base=0.366) | lr=3.0e-04
  [ 10] train_loss=0.1470 | val_pw=0.1504 | val_F1@15=0.382 (base=0.366) | lr=3.0e-04
  [ 20] train_loss=0.1348 | val_pw=0.1543 | val_F1@15=0.406 (base=0.366) | lr=2.9e-04
  [ 30] train_loss=0.1257 | val_pw=0.1760 | val_F1@15=0.389 (base=0.366) | lr=2.7e-04


KeyboardInterrupt: 

In [ ]:
# === PHASE 2: Joint Peak-Weighted RMSE + REINFORCE F1 ===
# Freeze encoder, fine-tune decoder only

for name, param in model.named_parameters():
    if any(k in name for k in ['enc1', 'enc2', 'enc3']):
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2: {trainable:,} trainable parameters (encoder frozen)')

optimizer2 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=1e-4, weight_decay=1e-3)

alpha_pw = 0.3   # peak-weighted RMSE weight
alpha_rl = 0.7   # REINFORCE weight
rl_baseline = None
best_val_f1 = 0.0
patience_counter = 0

print(f'=== PHASE 2: {alpha_pw:.0%} PW-RMSE + {alpha_rl:.0%} REINFORCE F1@15 ===')
for epoch in range(80):
    model.train()
    epoch_pw, epoch_rl, epoch_reward = [], [], []
    
    for pred, target, morgan, _ in train_loader:
        # Peak-weighted RMSE (differentiable)
        refined = model(pred, morgan)
        pw_loss = peak_weighted_rmse(refined, target)
        
        # REINFORCE F1 (non-differentiable reward)
        rl_loss, reward, rl_baseline = reinforce_f1_loss(
            model, pred, morgan, target, x_grid_np,
            K=5, baseline=rl_baseline, sigma=0.02, tol=15.0
        )
        
        loss = alpha_pw * pw_loss + alpha_rl * rl_loss
        
        optimizer2.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer2.step()
        
        epoch_pw.append(pw_loss.item())
        epoch_rl.append(rl_loss.item())
        epoch_reward.append(reward)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        metrics = evaluate(model, val_loader, x_grid_np)
        history['phase'].append(2)
        history['epoch'].append(150 + epoch)
        history['train_loss'].append(np.mean(epoch_pw))
        history['val_pw'].append(metrics['pw_rmse'])
        history['val_f1'].append(metrics['f1'])
        history['val_f1_base'].append(metrics['f1_baseline'])
        print(f'  [{epoch+1:3d}] pw={np.mean(epoch_pw):.4f} rl={np.mean(epoch_rl):.4f} '
              f'reward={np.mean(epoch_reward):.3f} | '
              f'val_F1@15={metrics["f1"]:.3f} (base={metrics["f1_baseline"]:.3f})')
        
        if metrics['f1'] > best_val_f1:
            best_val_f1 = metrics['f1']
            torch.save(model.state_dict(), 'refinement_phase2_best.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 15:
                print(f'  Early stopping at epoch {epoch+1}')
                break

model.load_state_dict(torch.load('refinement_phase2_best.pth', weights_only=True))
metrics = evaluate(model, val_loader, x_grid_np)
print(f'\nPhase 2 best: F1@15={metrics["f1"]:.3f} (base={metrics["f1_baseline"]:.3f})')

In [ ]:
# === Final evaluation on test set ===

test_metrics = evaluate(model, test_loader, x_grid_np)

# Also compute at multiple tolerances
model.eval()
all_refined, all_target, all_input = [], [], []
with torch.no_grad():
    for pred, target, morgan, _ in test_loader:
        refined = model(pred, morgan)
        all_refined.append(refined.numpy())
        all_target.append(target.numpy())
        all_input.append(pred.numpy())

all_refined = np.concatenate(all_refined)
all_target = np.concatenate(all_target)
all_input = np.concatenate(all_input)

print('=== TEST SET RESULTS ===')
print(f'{"Metric":<12} {"Input (raw)":>14} {"Refined":>14} {"Delta":>10}')
print('-' * 52)
for tol in [5, 10, 15, 20]:
    f1_raw = batch_f1(all_input, all_target, x_grid_np, tol=tol).mean()
    f1_ref = batch_f1(all_refined, all_target, x_grid_np, tol=tol).mean()
    delta = f1_ref - f1_raw
    print(f'F1@{tol:<8} {f1_raw:>14.3f} {f1_ref:>14.3f} {delta:>+9.3f}')

# Cosine similarity
cos_raw = np.mean([np.dot(all_input[i], all_target[i]) / (np.linalg.norm(all_input[i]) * np.linalg.norm(all_target[i]) + 1e-8)
                    for i in range(len(all_input))])
cos_ref = np.mean([np.dot(all_refined[i], all_target[i]) / (np.linalg.norm(all_refined[i]) * np.linalg.norm(all_target[i]) + 1e-8)
                    for i in range(len(all_refined))])
print(f'{"Cosine":<12} {cos_raw:>14.3f} {cos_ref:>14.3f} {cos_ref - cos_raw:>+9.3f}')

In [ ]:
# === Visualization: before/after for 6 test molecules ===
WONG = {'blue': '#0072B2', 'vermillion': '#D55E00', 'green': '#009E73', 'orange': '#E69F00'}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    if i >= len(all_input): break
    ax.plot(x_grid_np, all_target[i], color='k', lw=1.5, alpha=0.7, label='DFT reference')
    ax.plot(x_grid_np, all_input[i], color=WONG['orange'], lw=1, alpha=0.5, label='SpectraLoRA (raw)')
    ax.plot(x_grid_np, all_refined[i], color=WONG['blue'], lw=1.2, label='Refined')
    
    f1_raw = batch_f1(all_input[i:i+1], all_target[i:i+1], x_grid_np, tol=15)[0]
    f1_ref = batch_f1(all_refined[i:i+1], all_target[i:i+1], x_grid_np, tol=15)[0]
    ax.set_title(f'F1@15: {f1_raw:.2f} → {f1_ref:.2f}', fontsize=10)
    ax.set_xlim(500, 2000)
    if i == 0: ax.legend(fontsize=8)

fig.suptitle('Spectral Refinement: Before vs After', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === Training curves ===
h = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(h['epoch'], h['train_loss'], 'o-', markersize=3, label='Train PW-RMSE')
ax1.plot(h['epoch'], h['val_pw'], 's-', markersize=3, label='Val PW-RMSE')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Peak-Weighted RMSE')
ax1.legend(); ax1.set_title('Loss')

ax2.plot(h['epoch'], h['val_f1'], 'o-', markersize=3, color=WONG['blue'], label='Refined F1@15')
ax2.plot(h['epoch'], h['val_f1_base'], 's--', markersize=3, color=WONG['orange'], label='Raw input F1@15')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1@15')
ax2.legend(); ax2.set_title('Validation F1')

# Mark phase boundary
for ax in [ax1, ax2]:
    phase2_start = h[h['phase']==2]['epoch'].min() if 2 in h['phase'].values else None
    if phase2_start:
        ax.axvline(phase2_start, color='gray', ls=':', alpha=0.5, label='Phase 2 start')

plt.tight_layout()
plt.show()